In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Part 1: Multiple regression

The dataset contains data on test performance, school characteristics, and student demographic backgrounds from Californian districts (1998–1999). The objective is to predict test scores (`testscr`) based on sociodemographic variables.

**Q1.** How many Californian districts are represented in the dataset?

In [ ]:
df = pd.read_csv('../datasets/Caschool.csv', sep=';')
print('Number of districts:', df.shape[0])
display(df.head(5))

**Q2.** Using multiple regression, find which variables are significantly associated with test scores. Are the effects significant in the expected directions? (`testscr`)

In [ ]:
X = df.drop(columns=['testscr', 'readscr', 'mathscr', 'district', 'school', 'county', 'grades'])
y = df['testscr']
X_const = sm.add_constant(X)
model_full = sm.OLS(y, X_const).fit()
print(model_full.summary())

**Q3.** Is the variable `calwpct` significant when using a simple linear regression (one predictive variable only)? Explain the difference with the result of multiple regression.

In [ ]:
model_simple = sm.OLS(y, sm.add_constant(df[['calwpct']])).fit()
print(model_simple.summary())

**Q4.** Using model selection based on an information criterion (e.g. with the `aic` command in R), find a parsimonious regression model that explains the test score. Explain the computational procedure and provide the parsimonious model.

In [ ]:
# (Q4) Stepwise selection code here


**Q5.** Train the complete model (all variables) on all districts except the first 100 districts and evaluate the mean square prediction error on the first 100 districts.

In [ ]:
X_train, y_train = X.iloc[100:], y.iloc[100:]
X_test, y_test   = X.iloc[:100], y.iloc[:100]
m_full = sm.OLS(y_train, sm.add_constant(X_train)).fit()
y_pred = m_full.predict(sm.add_constant(X_test))
mse_full = np.mean((y_test - y_pred)**2)
print('MSE (full model):', mse_full)

**Q6.** Plot the first 100 fitted values as a function of the true value of the test score and evaluate if the model over- or underestimates test scores. Confirm your results using numerical computations.

In [ ]:
plt.scatter(y_test, y_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('True Score')
plt.ylabel('Predicted Score')
plt.show()
print('Mean error:', np.mean(y_pred - y_test))

**Q7.** Compute the mean square prediction error using the parsimonious model found in Q4. Compare prediction accuracies and evaluate if the reduced model under- or overestimates test score.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(numeric_cols, fontsize=9)
plt.colorbar(im, ax=ax)
ax.set_title('Correlation matrix — Caschool')
plt.tight_layout()
plt.show()

In [ ]:
# Select response variable
if 'testscr' in numeric_cols:
    response = 'testscr'
else:
    response = numeric_cols[-1]

predictors = [c for c in numeric_cols if c != response]

X = sm.add_constant(df[predictors].dropna())
y = df.loc[X.index, response]

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
fitted = model.fittedvalues
residuals = model.resid

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(fitted, residuals, alpha=0.5, edgecolors='k', linewidths=0.3)
ax.axhline(0, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Fitted values')
ax.set_ylabel('Residuals')
ax.set_title('Residuals vs Fitted values')
plt.tight_layout()
plt.show()

## Your answers here
